In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from change_of_basis import cmat_of_tmat, tmat_of_cmat
from christoffel import ChristoffelMatrix
from materials import get_materials_Cvec
from ray_theory import group_arrival_times
from safe_module import closest
from utilities import sm2v, v2sm, print_matrix

In [ ]:
write = False

In [ ]:
# select case
# case 1 - station showing High Variability across SymGroups
# case 2 - station showing Low Variability across SymGroups
# case 3 - station showing High Variability across t20 models
# case 4 - station showing Low Variability across t20 models
# case 5 - station showing High Variability across t2_5 models
# case 6 - station showing Low Variability across t2_5 models

case = 1
mark_group_arrivals = True

In [ ]:
map = "Brown"

In [ ]:
if case==1:
    # station showing High Variability across SymGroups
    database_index = 0
    stn = "R33-25"
    x_max = 51
    scale = 1.3

elif case==2:
    # station showing Low Variability across SymGroups
    database_index = 0
    stn = "R18-17"
    x_max = 41
    scale = 1

elif case==3:
    # station showing High Variability across t20 models
    database_index = 1
    stn = "R33-25"
    x_max = 41
    scale = 1

elif case==4:
    # station showing Low Variability across t20 models
    database_index = 1
    stn = "R18-17"
    x_max = 31
    scale = 1

elif case==5:
    # station showing High Variability across t2_5 models
    database_index = 2
    stn = "R33-25"
    x_max = 41
    scale = 1

elif case==6:
    # station showing Low Variability across t2_5 models
    database_index = 2
    stn = "R18-17"
    x_max = 31
    scale = 1

component = 'Z'
density = 2623

sampling_rate = 250
n_samples = (x_max+1) * sampling_rate

source_coords = np.array([0, 0, -75000])

databases = ["SymGroups", "t20", "t2_5"]
database_type = databases[database_index]
models = [["ISO", "ISO_XISO", "XISO", "XISO_TET", "TET", "TET_ORTH", "ORTH", "ORTH_MONO", "MONO", "MONO_TRIV", "TRIV"],
          ["t00", "t20", "t40", "t60", "t80", "t100"],
          ["t00", "t2_5", "t5", "t7_5", "t10", "t12_5", "t15", "t17_5", "t20"]][database_index]

In [ ]:
# direction cosine of a point with respect to another point in 3D

def direction_cosine(p1, p2):
    return (p2-p1) / np.linalg.norm(p2-p1)

In [ ]:
# get arrival times at a station using the Christoffel matrix

def get_arrival_times(n,Cvec,print_times=False):
    
    for i, value in enumerate(Cvec):
        C[list(C.keys())[i]] = value / density
    
    gamma_matrix = ChristoffelMatrix(n, C)
    eigenvalues, eigenvectors = np.linalg.eig(gamma_matrix)
    arrival_times = ((np.linalg.norm(source_coords - station_coords)) / np.sqrt(eigenvalues))
    arrival_times.sort()
    
    if print_times:
        print(f"t1 = {arrival_times[0]:.1f}s")
        print(f"t2 = {arrival_times[1]:.1f}s")
        print(f"t3 = {arrival_times[2]:.1f}s")
    
    return arrival_times

In [ ]:
# read the stations files to find the x, y and z coordinates of the station

root_dir = Path.cwd().resolve().parent.parent
stations_file = root_dir / "Scripts" / "Data" / "SPECFEM3D_CARTESIAN" / "DATA" / "STATIONS"

station_data = np.loadtxt(stations_file, dtype=str)

for station in station_data:
    if station[0] == stn:
        station_coords = np.array([float(station[3]), float(station[2]), -1*float(station[4])])
        break

In [ ]:
# initialize a dictionary C with random values

C = {
    "C11": 1, "C12": 0.2, "C13": 0.3, "C14": 0.1, "C15": 0.1, "C16": 0.2,
    "C22": 1, "C23": 0.2, "C24": 0.1, "C25": 0.2, "C26": 0.3,
    "C33": 1, "C34": 0.2, "C35": 0.2, "C36": 0.1,
    "C44": 0.8, "C45": 0.3, "C46": 0.2,
    "C55": 0.9, "C56": 0.1,
    "C66": 0.7
    }

In [ ]:
# read and store a list of Voigt matrices in vector form

Cvec, *_ = get_materials_Cvec(material=map)
Cvec = Cvec * 10**9
c_vecs = []

Tmat = tmat_of_cmat(v2sm(Cvec))

if database_type == "SymGroups":
    t_mats = [closest(Tmat, "ISO"),
              Tmat,
              closest(Tmat, "XISO"),
              Tmat,
              closest(Tmat, "TET"),
              Tmat,
              closest(Tmat, "ORTH"),
              Tmat,
              closest(Tmat, "MONO"),
              Tmat,
              Tmat]
    
    for i in [1, 3, 5, 7, 9]:
        t_mats[i] = 0.5 * (t_mats[i-1] + t_mats[i+1])
        
    for t_mat_test in t_mats:
        c_mat_test = cmat_of_tmat(t_mat_test)
        c_vec_test = sm2v(c_mat_test)
        c_vecs.append(c_vec_test)

elif database_type[0] == "t":
    if database_type == "t20":
        t_min  = 0
        t_max  = 1
        dt     = 1/5
    
    elif database_type == "t2_5":
        t_min  = 0 / 100
        t_max  = 20 / 100
        dt     = 2.5 / 100
    
    t_mat1 = closest(Tmat, 'ISO')
    t_mat2 = Tmat
    
    npts     = round( (t_max - t_min) / dt + 1 )
    frac     = np.linspace(t_min,t_max,npts)
    
    for i in range(len(models)):
        t_mat_test = (1-frac[i]) * t_mat1 + (frac[i]) * t_mat2
        c_mat_test = cmat_of_tmat(t_mat_test)
        c_vec_test = sm2v(c_mat_test)
        c_vecs.append(c_vec_test)

In [ ]:
# compute the direction cosine of the station from the source

n = direction_cosine(source_coords, station_coords)

In [ ]:
# get path to home directory

home_path = Path.home()
database_path = home_path / "Downloads" / "BROWN" / f"{map}_{database_type}"

In [ ]:
# compute the max amplitude among all the seismograms to be plotted in the 
# record section

y_global_max = 0
for i, model in enumerate(models):
    path = database_path / f"{model}" / f"XX.{stn}.CX{component}.semd"
    data = np.loadtxt(path)
    x = data[:, 0]
    y = data[:, 1]
    y = np.gradient(y, x)
    y_local_max = np.max(abs(y))
    if y_local_max > y_global_max:
        y_global_max = y_local_max

In [ ]:
# plotting the record section

spacing_between_seismograms = 1
    
fig, axs = plt.subplots(1, 1, figsize=(6, 15))
for i, model in enumerate(models):
    
    print(f"running model {model}")
    
    path = database_path / f"{model}" / f"XX.{stn}.CX{component}.semd"
    depth = i * spacing_between_seismograms
        
    data = np.loadtxt(path)
    x = data[:, 0]
    y = data[:, 1]
    y = np.gradient(y, x)
    y = (y / y_global_max) * scale - depth
    axs.plot(x[:n_samples], y[:n_samples], c='k', linewidth=0.5)

    if mark_group_arrivals:
        c_vec = c_vecs[i]
        c_mat = v2sm(c_vec)
        # arrival_times = get_arrival_times(n, c_vecs[i])
        arrival_times = group_arrival_times(c_mat, density, source_coords, station_coords)

        for x in arrival_times:
            axs.vlines(x=x, ymin=-0.25-depth, ymax=0.25-depth, color='b',
                       linestyle='--', alpha=0.3)
        
    axs.set_xlim(-2, x_max)

axs.set_xlabel("Time (s)")
axs.set_yticks([])
    
if write:
    if mark_group_arrivals:
        plt.savefig(f"record_section_brown_{database_type}_{stn}_group_arrivals.pdf", format="pdf", bbox_inches='tight')
    else:
        plt.savefig(f"record_section_brown_{database_type}_{stn}.pdf", format="pdf", bbox_inches='tight')
plt.show()

In [ ]:
# zoomed in standalone plots for each seismogram

l = len(models)
fig, axs = plt.subplots(l, 1, figsize=(10, 7*l))
for i, model in enumerate(models):
    
    path = database_path / f"{model}" / f"XX.{stn}.CX{component}.semd"
        
    data = np.loadtxt(path)
    x = data[:, 0]
    y = data[:, 1]
    # differentiate y with respect to x
    y = np.gradient(y, x)
    y_max = np.max(abs(y))
    axs[i].plot(x, y, c='k', label = f"component = {component}")
    
    arrival_times = get_arrival_times(n, c_vecs[i])
    for x in arrival_times: 
        # plot vertical lines at given x values
        axs[i].axvline(x=x, color='b', linestyle='--')
    axs[i].legend()
    
    axs[i].set_title(f"station = {stn}, model = {model}")
    axs[i].set_xlim(-2, x_max)
    axs[i].set_ylim(-1.1*y_max, 1.1*y_max)
 
plt.show()

In [ ]:
# zoomed in standalone plots for each component of a selected seismogram

components = ['X', 'Y', 'Z']

stn = stn
model = model

path = database_path / f"{model}"
paths = [path / f"XX.{stn}.CX{comp}.semd" for comp in components]

fig, axs = plt.subplots(3, 1, figsize=(10, 21))
for i, path in enumerate(paths):
    
    data = np.loadtxt(path)
    x = data[:, 0]
    y = data[:, 1]
    # differentiate y with respect to x
    y = np.gradient(y, x)
    y_max = np.max(abs(y))
    axs[i].plot(x, y, c='k', label = f"component = {components[i]}")
    
    c_vec = c_vecs[models.index(model)]
    c_mat = v2sm(c_vec)
    # arrival_times = get_arrival_times(n, c_vec)
    arrival_times = group_arrival_times(c_mat, density, source_coords, station_coords)
    
    for x in arrival_times: 
        # plot vertical lines at given x values
        axs[i].axvline(x=x, color='r', linestyle='--')
    axs[i].legend()

    axs[i].set_title(f"station = {stn}, model = {model}")
    axs[i].set_xlim(-2, x_max)
    axs[i].set_ylim(-1.1*y_max, 1.1*y_max)

plt.show()